# Notebook 4 — Detection Evaluation & Final Taux
### Did the ontology-driven autoencoders catch the three events?

This notebook turns the trained detection models (Notebook 3) into the **headline result**:

1. **ROC / PR curves** — how well per-window reconstruction error separates event windows from normal.
2. **Per-event detection** — recall on the *fire*, *dust/humidity* and *power-outage* windows, plus the
   false-positive rate on normal windows.
3. **Localisation** — per-**node** reconstruction error on each event, showing *which variable* drove
   the detection (the payoff of the ontology graph: it points at the culprit).
4. **The taux** — the final event-detection rate.

It reloads the trained autoencoders and recomputes per-node and per-window errors, so it is
self-contained from the saved models + graph (needs torch + torch_geometric, like Notebook 3).


In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GINConv, TransformerConv, BatchNorm, LayerNorm

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Load graph + per-deployment standardisation (matches Notebook 3)

In [ ]:
_C = [Path("processed/graph"), Path("processed"), Path(".")]
GP = next((p for p in _C if (p / "graph_windows.npz").exists()), Path("processed/graph"))
d = np.load(GP / "graph_windows.npz", allow_pickle=True)
man = json.load(open(GP / "graph_manifest.json"))

X_raw = d["X_raw"]; edge_index_np = d["edge_index"]
event_id = d["event_id"]; anomaly = d["anomaly"]; location = d["location"].astype(str)
det_train = d["det_train"]; det_val = d["det_val"]; det_test = d["det_test"]
W, N_NODES, F_CH = X_raw.shape
VARS = man["node_order"]; FEATURES = man["feature_channels"]
EVENT_ID = man["event_id_map"]; ID_EVENT = {v: k for k, v in EVENT_ID.items()}

X = X_raw.copy()
for loc in ["laboratory", "apartment"]:
    lm = location == loc; tm = lm & det_train
    if tm.sum() == 0: continue
    mu = X_raw[tm][:, :, :7].mean(0); sd = X_raw[tm][:, :, :7].std(0); sd[sd < 1e-8] = 1.0
    X[lm, :, :7] = (X_raw[lm][:, :, :7] - mu) / sd
X = X.astype(np.float32)

edge_index = torch.tensor(edge_index_np, dtype=torch.long)
data_all = [Data(x=torch.tensor(X[w], dtype=torch.float), edge_index=edge_index) for w in range(W)]
all_loader = DataLoader(data_all, batch_size=256, shuffle=False)
print(f"Windows={W} Nodes={N_NODES} | events in test:",
      {ID_EVENT[i]: int(((event_id == i) & det_test).sum()) for i in [1, 2, 3]})

## 2. Reload the trained autoencoders (same architecture as Notebook 3)

In [ ]:
class GraphAE(nn.Module):
    def __init__(self, in_ch, hid=64, latent=16, conv="gcn", nl=2, heads=4, dr=0.2, pe=None):
        super().__init__()
        self.conv_type = conv; self.dr = dr
        extra = pe.shape[1] if (conv == "sat" and pe is not None) else 0
        if conv == "sat": self.register_buffer("pe", pe)
        self.enc_in = nn.Linear(in_ch + extra, hid)
        self.convs = nn.ModuleList(); self.norms = nn.ModuleList()
        for _ in range(nl):
            if conv == "gcn":
                self.convs.append(GCNConv(hid, hid, add_self_loops=True)); self.norms.append(BatchNorm(hid))
            elif conv == "gin":
                mlp = nn.Sequential(nn.Linear(hid, 2 * hid), nn.BatchNorm1d(2 * hid), nn.ReLU(), nn.Linear(2 * hid, hid))
                self.convs.append(GINConv(mlp, train_eps=True)); self.norms.append(BatchNorm(hid))
            else:
                self.convs.append(TransformerConv(hid, hid // heads, heads=heads, dropout=dr, concat=True))
                self.norms.append(LayerNorm(hid))
        self.to_latent = nn.Linear(hid, latent)
        self.dec = nn.Sequential(nn.Linear(latent, hid), nn.ReLU(), nn.Dropout(dr), nn.Linear(hid, in_ch))

    def forward(self, x, ei):
        h = x
        if self.conv_type == "sat":
            B = x.size(0) // self.pe.size(0); h = torch.cat([x, self.pe.repeat(B, 1)], dim=-1)
        h = F.relu(self.enc_in(h)); h = F.dropout(h, self.dr, self.training)
        for c, n in zip(self.convs, self.norms):
            h = F.relu(n(c(h, ei))); h = F.dropout(h, self.dr, self.training)
        return self.dec(self.to_latent(h))

def degree_pe(ei, n):
    deg = torch.zeros(n); deg.scatter_add_(0, ei[0], torch.ones(ei.size(1)))
    return (deg / max(deg.max().item(), 1.0)).unsqueeze(1)
def rwse_pe(ei, n, k=8):
    deg = torch.zeros(n); deg.scatter_add_(0, ei[0], torch.ones(ei.size(1)))
    val = (1.0 / (deg + 1e-8))[ei[0]]; P = torch.sparse_coo_tensor(ei, val, (n, n)).coalesce().to_dense()
    rw = torch.zeros(n, k); Pk = P.clone()
    for i in range(k): rw[:, i] = Pk.diagonal(); Pk = Pk @ P
    return rw
PE = torch.cat([degree_pe(edge_index, N_NODES), rwse_pe(edge_index, N_NODES, 8)], dim=1)

CONFIGS = {"GCN": dict(conv="gcn"), "GIN": dict(conv="gin"), "SAT-Graph": dict(conv="sat", heads=4)}
MODELS = {}
for name, kw in CONFIGS.items():
    pe = PE if kw["conv"] == "sat" else None
    m = GraphAE(F_CH, hid=64, latent=16, nl=2, pe=pe, **kw).to(DEVICE)
    m.load_state_dict(torch.load(GP / f"{name.lower().replace('-', '_')}_ae.pt", map_location=DEVICE))
    m.eval(); MODELS[name] = m
print("Reloaded:", list(MODELS))

## 3. Per-node and per-window reconstruction errors

In [ ]:
@torch.no_grad()
def node_errors(model):
    out = []
    for b in all_loader:
        b = b.to(DEVICE)
        e = ((model(b.x, b.edge_index) - b.x) ** 2).mean(dim=1)     # per node
        out.append(e.view(-1, N_NODES).cpu().numpy())               # (B, N)
    return np.concatenate(out, axis=0)                              # (W, N)

E_NODE = {name: node_errors(m) for name, m in MODELS.items()}       # per model: (W, N)
E_WIN = {name: e.mean(axis=1) for name, e in E_NODE.items()}        # per model: (W,)
print("Computed per-node + per-window errors for", list(E_WIN))

## 4. ROC & PR curves (per-window error vs anomaly label, test set)

In [ ]:
def roc_points(y, s):
    o = np.argsort(-s); y = y[o]; tp = np.cumsum(y); fp = np.cumsum(1 - y)
    P = max(y.sum(), 1); Nn = max(len(y) - y.sum(), 1)
    return np.r_[0, fp / Nn], np.r_[0, tp / P]
def pr_points(y, s):
    o = np.argsort(-s); y = y[o]; tp = np.cumsum(y); fp = np.cumsum(1 - y)
    return tp / max(y.sum(), 1), tp / (tp + fp)
def auc(x, yv):
    x = np.asarray(x); yv = np.asarray(yv)
    return float(np.sum(np.diff(x) * (yv[1:] + yv[:-1]) / 2.0))
def ap_score(y, s):
    y = np.asarray(y); s = np.asarray(s)
    if y.sum() == 0: return float("nan")
    o = np.argsort(-s); y = y[o]; tp = np.cumsum(y); fp = np.cumsum(1 - y)
    prec = tp / (tp + fp); rec = tp / y.sum(); rp = np.r_[0, rec[:-1]]
    return float(((rec - rp) * prec).sum())

colors = {"GCN": "#378ADD", "GIN": "#1D9E75", "SAT-Graph": "#D85A30"}
te = det_test
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
auc_tbl = []
for name in MODELS:
    s = E_WIN[name][te]; y = anomaly[te]
    fpr, tpr = roc_points(y, s); rec, prec = pr_points(y, s)
    roc_auc = auc(fpr, tpr); pr_auc = ap_score(y, s)
    auc_tbl.append({"Model": name, "ROC_AUC": round(roc_auc, 4), "PR_AUC": round(pr_auc, 4)})
    ax1.plot(fpr, tpr, color=colors[name], lw=1.6, label=f"{name} (AUC={roc_auc:.3f})")
    ax2.plot(rec, prec, color=colors[name], lw=1.6, label=f"{name} (AP={pr_auc:.3f})")
ax1.plot([0, 1], [0, 1], "k--", lw=0.8); ax1.set_xlabel("false positive rate"); ax1.set_ylabel("true positive rate")
ax1.set_title("ROC — event windows vs normal"); ax1.legend()
ax2.set_xlabel("recall"); ax2.set_ylabel("precision"); ax2.set_title("Precision–Recall"); ax2.legend()
fig.tight_layout(); plt.savefig(GP / "nb04_roc_pr.png", dpi=130, bbox_inches="tight"); plt.show()
print(pd.DataFrame(auc_tbl).to_string(index=False))

## 5. Per-event detection + the final taux

The threshold is the 99th percentile of **normal validation** error (≈1% false positives on clean
data). For each model we report: per-event window recall, the false-positive rate on normal test
windows, whether each event was detected at all (≥1 window flagged), and the **taux** — the overall
fraction of event windows caught.


In [ ]:
THRESH_Q = 0.99
rows = []
for name in MODELS:
    err = E_WIN[name]; thr = float(np.quantile(err[det_val], THRESH_Q)); te = det_test
    norm_te = te & (anomaly == 0)
    row = {"Model": name, "threshold": round(thr, 5),
           "FPR_normal": round(float((err[norm_te] > thr).mean()), 4)}
    detected = 0; total_ev_win = 0; caught_ev_win = 0
    for eid in [1, 2, 3]:
        m = (event_id == eid) & te
        if m.sum() == 0:
            row[f"{ID_EVENT[eid]}_recall"] = None; continue
        flagged = err[m] > thr
        row[f"{ID_EVENT[eid]}_recall"] = round(float(flagged.mean()), 3)
        detected += int(flagged.any()); total_ev_win += int(m.sum()); caught_ev_win += int(flagged.sum())
    row["events_detected"] = f"{detected}/{sum((event_id==e).any() for e in [1,2,3])}"
    row["taux_window"] = round(caught_ev_win / max(total_ev_win, 1), 3)
    rows.append(row)

taux_df = pd.DataFrame(rows)
taux_df.to_csv(GP / "nb04_detection_taux.csv", index=False)
print("=== Detection performance + taux ===")
print(taux_df.to_string(index=False))

## 6. Localisation — which variable drove each detection?

For each event we average the per-node reconstruction error over that event's windows and express it
as a z-score against the node's normal-test error. High z = that variable reconstructed far worse
during the event, i.e. it is what the model flagged. We expect PM nodes to light up for July-9, and
NO2 for the fire.


In [ ]:
best = taux_df.sort_values("taux_window", ascending=False)["Model"].iloc[0]
EN = E_NODE[best]
norm_mu = EN[det_test & (anomaly == 0)].mean(0); norm_sd = EN[det_test & (anomaly == 0)].std(0); norm_sd[norm_sd < 1e-8] = 1.0

present = [eid for eid in [1, 2, 3] if (event_id == eid).any()]
Z = np.zeros((len(present), N_NODES))
for r, eid in enumerate(present):
    m = (event_id == eid) & det_test
    Z[r] = (EN[m].mean(0) - norm_mu) / norm_sd

# show the most-implicated variables across events
top = np.argsort(-Z.max(0))[:14]
fig, ax = plt.subplots(figsize=(12, 3.6))
im = ax.imshow(Z[:, top], aspect="auto", cmap="OrRd", vmin=0)
ax.set_xticks(range(len(top))); ax.set_xticklabels([VARS[i] for i in top], rotation=45, ha="right")
ax.set_yticks(range(len(present))); ax.set_yticklabels([ID_EVENT[e] for e in present])
ax.set_title(f"{best} — per-variable reconstruction error during events (z vs normal)")
fig.colorbar(im, ax=ax, shrink=0.8, label="z-score"); fig.tight_layout()
plt.savefig(GP / "nb04_localization.png", dpi=130, bbox_inches="tight"); plt.show()

print(f"Top reconstruction-error drivers per event ({best}):")
for r, eid in enumerate(present):
    order = np.argsort(-Z[r])[:5]
    print(f"  {ID_EVENT[eid]:14s}: " + ", ".join(f"{VARS[i]}(z={Z[r,i]:.1f})" for i in order))

## 7. Final summary

In [ ]:
best = taux_df.sort_values("taux_window", ascending=False)["Model"].iloc[0]
brow = taux_df[taux_df["Model"] == best].iloc[0]
summary = {
    "best_detection_model": best,
    "roc_auc": float(pd.DataFrame(auc_tbl).set_index("Model").loc[best, "ROC_AUC"]),
    "pr_auc": float(pd.DataFrame(auc_tbl).set_index("Model").loc[best, "PR_AUC"]),
    "taux_window_detection_rate": float(brow["taux_window"]),
    "events_detected": brow["events_detected"],
    "fpr_normal": float(brow["FPR_normal"]),
    "per_event_recall": {ID_EVENT[e]: (None if brow.get(f"{ID_EVENT[e]}_recall") is None
                                       else float(brow[f"{ID_EVENT[e]}_recall"])) for e in [1, 2, 3]},
}
json.dump(summary, open(GP / "nb04_detection_summary.json", "w"), indent=2, default=str)
print("=== HEADLINE DETECTION RESULT ===")
print(f"  Best model        : {best}")
print(f"  ROC-AUC / PR-AUC  : {summary['roc_auc']} / {summary['pr_auc']}")
print(f"  Events detected   : {summary['events_detected']}")
print(f"  Taux (window rate): {summary['taux_window_detection_rate']}")
print(f"  FPR on normal     : {summary['fpr_normal']}")
print("\nSaved: nb04_roc_pr.png, nb04_localization.png, nb04_detection_taux.csv, nb04_detection_summary.json")

## 8. Hand-off to Notebook 5

The detection headline is complete: ROC/PR, per-event recall, the localisation of the driving
variables, and the final taux. Remember these are **detection** numbers (reconstruction error on the
three events) — not forecasting accuracy.

**Next — Notebook 5: prediction comparison.** On the same ontology graph, the leakage-clean pollutant
regression: GNN (masked pollutant-node prediction) vs RNN / LSTM / GRU baselines, scored with
R²/MAE/RMSE and placed against Xuanzhe — the controlled comparison that complements the detection
headline. Notebook 6 then assembles both into the final write-up.
